In this notebook, I want to show how different types of prompting (zero-shot, few-shot, chain-of-thought) can influence the performances of LLMs on different tasks. I I will use FLAN-T5 (instruction-tuned) for this work.

In [18]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

## 1. Setup

In [2]:
# Load model

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\Chiara\anaconda3\envs\cv\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Chiara\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [3]:
# Function to generate output

def generate(prompt, max_new_tokens=50):

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
# Test

prompt = "Classify the sentiment: I love this movie"
print(generate(prompt))

positive


The model correctly execute the task.

## 2. Task design and Dataset

I select different task of increasing complexitys:
1. Sentiment analysis
2. Topic classification
3. Reasoning

According to the prompt, the tasks help evaluating if:
* The instructions are followed
* The outputs are consistent
* The model is capable of reasoning

In [6]:
# Sentiment
sentiment_data = [
    {"text": "I absolutely loved this movie, it was fantastic!", "label": "Positive"},
    {"text": "The film was boring and too long.", "label": "Negative"},
    {"text": "It was okay, not great but not bad either.", "label": "Neutral"},
    {"text": "Amazing performance by the actors!", "label": "Positive"},
    {"text": "I wouldn't recommend this to anyone.", "label": "Negative"},
    {"text": "The plot was predictable but enjoyable.", "label": "Neutral"},
]

# Topics
topic_data = [
    {"text": "The team won the championship after a thrilling final.", "label": "Sports"},
    {"text": "The government passed a new law regarding taxes.", "label": "Politics"},
    {"text": "New advancements in AI are transforming industries.", "label": "Technology"},
    {"text": "The player scored a hat-trick in the match.", "label": "Sports"},
    {"text": "The president gave a speech about economic reforms.", "label": "Politics"},
    {"text": "Quantum computing could revolutionize data processing.", "label": "Technology"},
]

# Reasoning
reasoning_data = [
    {"question": "If I have 3 apples and buy 2 more, how many apples do I have?", "answer": "5"},
    {"question": "John had 10 dollars and spent 4. How much does he have left?", "answer": "6"},
    {"question": "A box contains 6 red and 4 blue balls. How many balls are there in total?", "answer": "10"},
]

In [7]:
# Fromatter

def format_sentiment(example):
    return example["text"], example["label"]

def format_topic(example):
    return example["text"], example["label"]

def format_reasoning(example):
    return example["question"], example["answer"]

In [8]:
# Check

print(sentiment_data[0])
print(topic_data[0])
print(reasoning_data[0])

{'text': 'I absolutely loved this movie, it was fantastic!', 'label': 'Positive'}
{'text': 'The team won the championship after a thrilling final.', 'label': 'Sports'}
{'question': 'If I have 3 apples and buy 2 more, how many apples do I have?', 'answer': '5'}


## 3. Prompt engineering

I use different prompting strategies to see how they impact performance. I use:
* Zero-shot
* Few-shot
* Chain of thought

In [9]:
# Zero-shot on the 3 tasks

# sentiment
def zero_shot_sentiment(text):
    return f"""
Classify the sentiment as Positive, Negative, or Neutral.
Respond with only one word.

Text: {text}
Answer:
"""

# topic
def zero_shot_topic(text):
    return f"""
Classify the topic of the text into one of the following categories:
Sports, Politics, Technology.

Respond with only one word.

Text: {text}
Answer:
"""

# reasoning 
def zero_shot_reasoning(question):
    return f"""
Answer the following question.

Question: {question}
Answer:
"""

In [10]:
# Few-shot on the 3 tasks

# sentiment
def few_shot_sentiment(text):
    return f"""
Classify the sentiment as Positive, Negative, or Neutral.

Examples:
Text: I love this product → Positive
Text: This is terrible → Negative
Text: It is okay → Neutral

Now classify:

Text: {text}
Answer:
"""

# topic
def few_shot_topic(text):
    return f"""
Classify the topic into: Sports, Politics, Technology.

Examples:
Text: The team won the match → Sports
Text: The president signed a law → Politics
Text: AI is evolving rapidly → Technology

Now classify:

Text: {text}
Answer:
"""

# reasoning
def few_shot_reasoning(question):
    return f"""
Answer the questions.

Examples:
Q: If I have 2 apples and buy 3 more, how many do I have?
A: 5

Q: John had 5 dollars and spent 2. How much is left?
A: 3

Now answer:

Q: {question}
A:
"""

In [11]:
# CoT on the reasoning task

def cot_reasoning(question):
    return f"""
Solve the problem step by step.

Question: {question}

Answer:
"""

# hidden
def cot_hidden_reasoning(question):
    return f"""
Solve the problem step by step, but provide only the final answer.

Question: {question}

Answer:
"""

In [12]:
# Prompt wrapper

def build_prompt(task, strategy, input_text):

    if task == "sentiment":
        if strategy == "zero":
            return zero_shot_sentiment(input_text)
        elif strategy == "few":
            return few_shot_sentiment(input_text)

    if task == "topic":
        if strategy == "zero":
            return zero_shot_topic(input_text)
        elif strategy == "few":
            return few_shot_topic(input_text)

    elif task == "reasoning":
        if strategy == "zero":
            return zero_shot_reasoning(input_text)
        if strategy == "few":
            return few_shot_reasoning(input_text)
        if strategy == "cot":
            return cot_reasoning(input_text)
        if strategy == "hidden_cot":
            return cot_hidden_reasoning(input_text)

    raise ValueError("Invalid task or strategy")

In [14]:
# Test zero-shot

text = "The movie was terrible"

prompt = zero_shot_sentiment(text)
print(generate(prompt))

Negative


In [15]:
# Test few-shot

prompt = few_shot_sentiment(text)
print(generate(prompt))

Negative


In [16]:
# Test CoT

q = "If I have 3 apples and buy 2 more, how many apples do I have?"

prompt = cot_reasoning(q)
print(generate(prompt))

I have 3 + 2 = 6 apples. I have 6 + 6 = 16 apples. The final answer: 16.


In [17]:
# Test hidden CoT

q = "If I have 3 apples and buy 2 more, how many apples do I have?"

prompt = cot_hidden_reasoning(q)
print(generate(prompt))

3 apples


## 4. Experiments

How do different prompt strategies perform?

In [31]:
# Sentiment analysis

results = []

for example in tqdm(sentiment_data):

    text = example["text"]
    true_label = example["label"]

    for strategy in ["zero", "few"]:
        prompt = build_prompt("sentiment", strategy, text)
        prediction = generate(prompt)

        results.append({
            "task": "sentiment",
            "input": text,
            "true_label": true_label,
            "prediction": prediction,
            "strategy": strategy
        })

100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 12.32it/s]


In [32]:
results[:1]

[{'task': 'sentiment',
  'input': 'I absolutely loved this movie, it was fantastic!',
  'true_label': 'Positive',
  'prediction': 'Positive',
  'strategy': 'zero'}]

In [33]:
# Topic classification

for example in tqdm(topic_data):

    text = example["text"]
    true_label = example["label"]

    for strategy in ["zero", "few"]:
        prompt = build_prompt("topic", strategy, text)
        prediction = generate(prompt)

        results.append({
            "task": "topic",
            "input": text,
            "true_label": true_label,
            "prediction": prediction,
            "strategy": strategy
        })

100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 12.42it/s]


In [34]:
# Reasoning

for example in tqdm(reasoning_data):

    question = example["question"]
    true_answer = example["answer"]

    for strategy in ["zero", "few", "hidden_cot"]:
        prompt = build_prompt("reasoning", strategy, question)
        prediction = generate(prompt)

        results.append({
            "task": "reasoning",
            "input": question,
            "true_label": true_answer,
            "prediction": prediction,
            "strategy": strategy
        })

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00,  7.07it/s]


In [35]:
# Create df

df = pd.DataFrame(results)
df

,task,input,true_label,prediction,strategy
0,sentiment,"I absolutely loved this movie, it was fantastic!",Positive,Positive,zero
1,sentiment,"I absolutely loved this movie, it was fantastic!",Positive,Positive,few
2,sentiment,The film was boring and too long.,Negative,Negative,zero
3,sentiment,The film was boring and too long.,Negative,Negative,few
4,sentiment,"It was okay, not great but not bad either.",Neutral,Negative,zero
5,sentiment,"It was okay, not great but not bad either.",Neutral,Positive,few
6,sentiment,Amazing performance by the actors!,Positive,Positive,zero
7,sentiment,Amazing performance by the actors!,Positive,Positive,few
8,sentiment,I wouldn't recommend this to anyone.,Negative,Negative,zero
9,sentiment,I wouldn't recommend this to anyone.,Negative,Negative,few


In [40]:
# Clean (good practise)

def clean_prediction(pred):
    return pred.strip().split("\n")[0].strip()

df["prediction_clean"] = df["prediction"].apply(clean_prediction)

Some results:

* Sentiment analysis task: results are usually in accordance with the true labels, and both zero-shot and few-shot prompting accurately predict positive and negative sentiments; however, they both fail in predicting neutral sentiment.
* Topic detection task: zero-shot predictions are always correct, while few-shot usually makes misclassifications.
* Reasoning task: results given by the 3 prompting methods are mostly consistent, but different with respect to the true label, meaning that the model fails in this kind of task, independently from the prompting technique.

## 5. Evaluation

I evaluate performance across different strategies:
* Accuracy for classification tasks
* Exact match for reasoning task

In [41]:
# Normalize labels (good practise)

def normalize(text):
    return text.lower().strip().replace(".", "")

df["pred_norm"] = df["prediction_clean"].apply(normalize)
df["true_norm"] = df["true_label"].apply(normalize)

In [42]:
# Accuracy

def compute_accuracy(dataframe):
    return(dataframe["pred_norm"] == dataframe["true_norm"]).mean()

In [43]:
# Sentiment
sentiment_df = df[df["task"] == "sentiment"]
sentiment_results = sentiment_df.groupby("strategy").apply(compute_accuracy)
print("Sentiment accuracy:")
print(sentiment_results)

# Topic
topic_df = df[df["task"] == "topic"]
topic_results = topic_df.groupby("strategy").apply(compute_accuracy)
print("Topic accuracy:")
print(topic_results)

# Reasoning
reasoning_df = df[df["task"] == "reasoning"]
reasoning_results = reasoning_df.groupby("strategy").apply(compute_accuracy)
print("Reasoning Accuracy:")
print(reasoning_results)

Sentiment accuracy:
strategy
few     0.666667
zero    0.666667
dtype: float64
Topic accuracy:
strategy
few     0.333333
zero    1.000000
dtype: float64
Reasoning Accuracy:
strategy
few           0.0
hidden_cot    0.0
zero          0.0
dtype: float64


C:\Users\Chiara\AppData\Local\Temp\ipykernel_2752\1754125099.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentiment_results = sentiment_df.groupby("strategy").apply(compute_accuracy)
C:\Users\Chiara\AppData\Local\Temp\ipykernel_2752\1754125099.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  topic_results = topic_df.groupby("strategy").apply(compute_accuracy)
C:\Users\Chiara\AppData\Local\Temp\ipykernel_275

In [44]:
# Table

summary = pd.DataFrame({
    "Sentiment": sentiment_results,
    "Topic": topic_results,
    "Reasoning": reasoning_results  
})

summary

,Sentiment,Topic,Reasoning
strategy,,,
few,0.666667,0.333333,0.0
hidden_cot,NaN,NaN,0.0
zero,0.666667,1.000000,0.0


Some insights:
* Zero-shot gives an accuracy 1 for topic classification. A reason could be the small dataset or the well distinguished categories. Few-shot instead gives a worse accuracy. This can be due to too simplistic examples.
* In the sentiment analysis task, the few-shot prompting doesn't perform better than zero-shot (it may be that the task is already simple enough).
* In the reasoning task, the model is not actually reasoning. All the answers are wrong, but mostly consistent across techniques.